In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END

In [31]:
model_id = "Bllossom/llama-3.2-Korean-Bllossom-3B"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype="auto"
)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

pipe = pipeline(
    task = "text-generation",
    model = model,
    tokenizer = tokenizer,
    do_sample = True,
    top_p = 0.9,
    temperature = 0.7,
    max_new_tokens = 64
)

llm = HuggingFacePipeline(pipeline=pipe)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cpu


In [32]:
template = "다음 문장에 대해 친절하게 대답해줘.\n\n{text}"
prompt = PromptTemplate.from_template(template)
parser = StrOutputParser()
hello_chain = prompt | llm | parser

In [43]:
def add_calculator(expression):
    numbers = expression.replace(" ", "").split("+")
    result = int(numbers[0]) + int(numbers[1])
    return result

In [44]:
def classify_question(state):
    text = state.get("user_input", "")

    if "+" in text:
        return {**state, "question_type": "calculator"}

    return {**state, "question_type": "hello"}

def hello_node(state):
    text = state.get("user_input", "")

    result = hello_chain.invoke(
        {"text": text}
    )

    return {**state, "response": result}

def calculator_node(state):
    text = state.get("user_input", "")
    result = add_calculator(text)
    return {**state, "response": result}

def route_by_type(state):
    return state.get("question_type", "hello")

In [45]:
graph = StateGraph(dict)

graph.add_node("classification", classify_question)
graph.add_node("hello", hello_node)
graph.add_node("calculator", calculator_node)

graph.set_entry_point("classification")

graph.add_conditional_edges(
    "classification",
    route_by_type,
    {
        "hello": "hello",
        "calculator": "calculator",
    }
)

graph.add_edge("hello", END)
graph.add_edge("calculator", END)

app = graph.compile()

In [46]:
user_input = "21 + 5"
result = app.invoke({"user_input": user_input})
print("응답:", result["response"])

응답: 26


In [42]:
user_input = "안녕"
result = app.invoke({"user_input": user_input})
print("응답:", result["response"])

응답: 다음 문장에 대해 친절하게 대답해줘.

안녕하서, 저는 한국에서 오래된 전통을 자랑하는 도시가 되는 서울에서 살고 있습니다. 서울은 한국의 중심부에 위치해 있으며, 다양한 문화와 역사적인 유산을 자랑합니다. 서울은 한국의 정치, 경제, 문화, 사회 등 모든 분야에서 중요한
